In [ ]:
!pip install pytrends yfinance pandas matplotlib -q

In [ ]:
# ============================================================
# GOOGLE TRENDS DATA EXTRACTION BY STOCK TICKER
# ============================================================

import pandas as pd
import yfinance as yf
import time
from datetime import datetime, timezone
from pytrends.request import TrendReq


In [ ]:
from google.colab import files
from pathlib import Path

In [ ]:
from google.colab import drive
import os
import shutil

# Attempt to unmount Google Drive if it's in an inconsistent state
try:
    drive.flush_and_unmount()
    print("Google Drive unmounted successfully for remount.")
except Exception as e:
    # If it's not mounted or an error occurs during unmount, proceed silently.
    print(f"Could not unmount Google Drive (possibly not mounted): {e}")

# Ensure the mount point directory is empty and ready
mountpoint = '/content/drive'
if os.path.exists(mountpoint) and os.listdir(mountpoint):
    print(f"Mountpoint {mountpoint} is not empty. Removing contents...")
    try:
        # Use shutil.rmtree to remove directory and its contents
        shutil.rmtree(mountpoint)
        print(f"Contents of {mountpoint} removed. Recreating directory...")
        os.makedirs(mountpoint, exist_ok=True)
        print(f"Directory {mountpoint} recreated.")
    except Exception as e:
        print(f"Error clearing or recreating mountpoint {mountpoint}: {e}")

# Now, attempt to mount the drive, forcing a remount if needed
drive.mount(mountpoint, force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Google Drive unmounted successfully for remount.
Mountpoint /content/drive is not empty. Removing contents...
Contents of /content/drive removed. Recreating directory...
Directory /content/drive recreated.
Mounted at /content/drive


In [ ]:
output_directory = Path("/content/drive/MyDrive/ColabNotebooks/DataForReputation")
output_directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ------------------------------------------------------------
# FUNCTION 1: Get company name from ticker
# ------------------------------------------------------------

def get_company_name(ticker):
    """
    Uses Yahoo Finance to convert a stock ticker
    into a company name.

    Example:
        AAPL -> Apple Inc.
        MSFT -> Microsoft Corporation
    """

    ticker = ticker.upper().strip()

    try:
        stock = yf.Ticker(ticker)
        info = stock.get_info()

        company_name = (
            info.get("shortName")
            or info.get("longName")
            or ticker
        )

        return company_name

    except Exception as e:
        print(f"Could not automatically determine company name: {e}")
        return ticker



In [ ]:
# ------------------------------------------------------------
# FUNCTION 2: Download Google Trends data
# ------------------------------------------------------------

def get_google_trends(
    ticker,
    start_date,
    end_date,
    search_term,
    company_name,
    geo="US",
    max_retries=3
):
    """
    Downloads Google Trends Interest Over Time data.

    Parameters
    ----------
    ticker : str
        Stock ticker, e.g. AAPL

    start_date : str
        Start date YYYY-MM-DD

    end_date : str
        End date YYYY-MM-DD

    search_term : str
        Term actually submitted to Google Trends

    company_name : str
        Company name associated with ticker

    geo : str
        Geography.
        US = United States
        "" = Worldwide
    """

    timeframe = f"{start_date} {end_date}"

    # Connect to Google Trends
    pytrends = TrendReq(
        hl="en-US",
        tz=300,
        timeout=(10, 25)
    )

    # Retry logic in case Google temporarily rejects the request
    for attempt in range(max_retries):

        try:

            print("\nRequesting Google Trends data...")
            print(f"Ticker:       {ticker}")
            print(f"Company:      {company_name}")
            print(f"Search term:  {search_term}")
            print(f"Timeframe:    {timeframe}")
            print(f"Geography:    {geo if geo else 'Worldwide'}")

            # Build Google Trends request
            pytrends.build_payload(
                kw_list=[search_term],
                cat=0,
                timeframe=timeframe,
                geo=geo,
                gprop=""
            )

            # Get interest over time
            df = pytrends.interest_over_time()

            if df.empty:
                raise ValueError(
                    "Google Trends returned no data for this query."
                )

            break

        except Exception as e:

            print(
                f"Attempt {attempt + 1} failed: {e}"
            )

            if attempt == max_retries - 1:
                raise

            wait_time = 5 * (2 ** attempt)

            print(
                f"Retrying after {wait_time} seconds..."
            )

            time.sleep(wait_time)

    # --------------------------------------------------------
    # Clean dataset
    # --------------------------------------------------------

    df = df.reset_index()

    # Rename Google Trends value
    df = df.rename(
        columns={
            search_term: "trend_interest",
            "date": "date",
            "isPartial": "is_partial"
        }
    )

    # Add metadata columns
    df["ticker"] = ticker
    df["company_name"] = company_name
    df["search_term"] = search_term
    df["geo"] = geo if geo else "WORLD"
    df["source"] = "Google Trends"
    df["retrieved_at_utc"] = datetime.now(
        timezone.utc
    ).isoformat()

    # Reorder columns
    columns = [
        "date",
        "ticker",
        "company_name",
        "search_term",
        "trend_interest",
        "is_partial",
        "geo",
        "source",
        "retrieved_at_utc"
    ]

    # isPartial is not always returned
    available_columns = [
        c for c in columns if c in df.columns
    ]

    df = df[available_columns]

    return df

In [ ]:
# ============================================================
# USER INPUT
# ============================================================

print("=" * 60)
print("GOOGLE TRENDS COMPANY DATA EXTRACTION")
print("=" * 60)

ticker = input(
    "\nEnter stock ticker (example: AAPL): "
).strip().upper()

start_date = input(
    "Enter start date (YYYY-MM-DD): "
).strip()

end_date = input(
    "Enter end date (YYYY-MM-DD): "
).strip()


GOOGLE TRENDS COMPANY DATA EXTRACTION

Enter stock ticker (example: AAPL): AAPL
Enter start date (YYYY-MM-DD): 2026-01-01
Enter end date (YYYY-MM-DD): 2026-08-23


In [ ]:

# ------------------------------------------------------------
# Validate dates
# ------------------------------------------------------------

start_dt = pd.to_datetime(start_date)
end_dt = pd.to_datetime(end_date)

if start_dt > end_dt:
    raise ValueError(
        "Start date must be before end date."
    )


In [ ]:
########NO NO NO ###########
# ------------------------------------------------------------
# Detect company name
# ------------------------------------------------------------

company_name = get_company_name(ticker)

print(
    f"\nDetected company: {company_name}"
)


# ------------------------------------------------------------
# Allow manual Google Trends search term
# ------------------------------------------------------------

print(
    "\nGoogle Trends usually works better with the company "
    "or brand name than with the ticker."
)

custom_search = input(
    f"Search term [press Enter to use '{company_name}']: "
).strip()

if custom_search:
    search_term = custom_search
else:
    search_term = company_name




Detected company: Apple Inc.

Google Trends usually works better with the company or brand name than with the ticker.
Search term [press Enter to use 'Apple Inc.']: 


In [ ]:
# ------------------------------------------------------------
# Geography
# ------------------------------------------------------------

geo = input(
    "Geography [US = United States; WORLD = worldwide; default US]: "
).strip().upper()

if geo == "":
    geo = "US"

elif geo == "WORLD":
    geo = ""


# ============================================================
# DOWNLOAD DATA
# ============================================================

trends_df = get_google_trends(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    search_term=search_term,
    company_name=company_name,
    geo=geo
)



Geography [US = United States; WORLD = worldwide; default US]: 

Requesting Google Trends data...
Ticker:       AAPL
Company:      Apple Inc.
Search term:  Apple Inc.
Timeframe:    2026-01-01 2026-08-23
Geography:    US
Attempt 1 failed: The request failed: Google returned a response with code 429
Retrying after 5 seconds...

Requesting Google Trends data...
Ticker:       AAPL
Company:      Apple Inc.
Search term:  Apple Inc.
Timeframe:    2026-01-01 2026-08-23
Geography:    US


/usr/local/lib/python3.13/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
